In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error


ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

print("Ratings Data:")
print(ratings.head())
print("-" * 30)
print("Movies Data:")
print(movies.head())

df = pd.merge(ratings, movies, on='movieId')
print("-" * 30)
print(f"Combined Dataset Shape: {df.shape}")

Ratings Data:
   userId  movieId  rating   timestamp
0       1       31     2.5  1260759144
1       1     1029     3.0  1260759179
2       1     1061     3.0  1260759182
3       1     1129     2.0  1260759185
4       1     1172     4.0  1260759205
------------------------------
Movies Data:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
------------------------------
Combined Dataset Shape: (100004, 6)


In [12]:
user_movie_matrix = df.pivot_table(index='userId', columns='title', values='rating')

user_movie_matrix.fillna(0, inplace=True)

print("User-Item Matrix Shape:", user_movie_matrix.shape)
print(user_movie_matrix.iloc[:5, :5]) 

User-Item Matrix Shape: (671, 9064)
title   "Great Performances" Cats (1998)  $9.99 (2008)  \
userId                                                   
1                                    0.0           0.0   
2                                    0.0           0.0   
3                                    0.0           0.0   
4                                    0.0           0.0   
5                                    0.0           0.0   

title   'Hellboy': The Seeds of Creation (2004)  \
userId                                            
1                                           0.0   
2                                           0.0   
3                                           0.0   
4                                           0.0   
5                                           0.0   

title   'Neath the Arizona Skies (1934)  'Round Midnight (1986)  
userId                                                           
1                                   0.0                     0.0  
2

In [13]:
user_similarity = cosine_similarity(user_movie_matrix)

user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

print("Similarity Matrix (User vs User):")
print(user_similarity_df.iloc[:5, :5])

Similarity Matrix (User vs User):
userId         1         2         3         4         5
userId                                                  
1       1.000000  0.000000  0.000000  0.074482  0.016818
2       0.000000  1.000000  0.124295  0.118821  0.103646
3       0.000000  0.124295  1.000000  0.081640  0.151531
4       0.074482  0.118821  0.081640  1.000000  0.130649
5       0.016818  0.103646  0.151531  0.130649  1.000000


In [14]:
def get_recommendations(user_id, num_recommendations=5):
   
    similar_users = user_similarity_df[user_id].sort_values(ascending=False).index[1:]
    
  
    user_seen_movies = user_movie_matrix.loc[user_id]
    seen_movies_list = user_seen_movies[user_seen_movies > 0].index.tolist()
    
    recommendations = {}
    
    
    for similar_user in similar_users[:10]:
        similar_user_ratings = user_movie_matrix.loc[similar_user]
        top_movies = similar_user_ratings[similar_user_ratings > 4].index.tolist()
        
        for movie in top_movies:
            if movie not in seen_movies_list:
                if movie in recommendations:
                    recommendations[movie] += 1 
                else:
                    recommendations[movie] = 1
    
    
    sorted_recs = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_recs[:num_recommendations]


target_user = 1
recs = get_recommendations(target_user)

print(f"Top Movie Recommendations for User {target_user}:")
for movie, score in recs:
    print(f"- {movie}")

Top Movie Recommendations for User 1:
- Godfather: Part II, The (1974)
- Godfather, The (1972)
- Alien (1979)
- Fantasia (1940)
- Star Wars: Episode IV - A New Hope (1977)


In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD

ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

df = pd.merge(ratings, movies, on='movieId')

user_movie_matrix = df.pivot_table(index='userId', columns='title', values='rating')
user_movie_matrix.fillna(0, inplace=True)

X = user_movie_matrix.T

SVD = TruncatedSVD(n_components=12, random_state=42)
matrix_decomposed = SVD.fit_transform(X)

corr_matrix = np.corrcoef(matrix_decomposed)

def recommend_similar_movie_svd(movie_title):
    movie_names = list(user_movie_matrix.columns)
    
    if movie_title not in movie_names:
        return "Movie not found."
        
    movie_idx = movie_names.index(movie_title)
    
    correlation_movie_ID = corr_matrix[movie_idx]
    
    recommend = list(user_movie_matrix.columns[(correlation_movie_ID > 0.9)])
    
    if movie_title in recommend:
        recommend.remove(movie_title)
        
    return recommend[:10]

test_movie = 'Star Wars (1977)'
recommendations = recommend_similar_movie_svd(test_movie)

print(f"Movies similar to '{test_movie}':")
for movie in recommendations:
    print(f"- {movie}")

Movies similar to 'Star Wars (1977)':
- M
- o
- v
- i
- e
-  
- n
- o
- t
-  
- f
- o
- u
- n
- d
- .
